EMBEDDING GENERATE

In [14]:
import pandas as pd

dataset = pd.read_csv("sentences.csv", header=None, names=["sentence"])
dataset.head(10)

,sentence
0,A little girl is smiling and running outside
1,A man is drawing on a digital dry erase board
2,A black bird is sitting on a dead tree
3,An elderly man is sitting on a bench
4,A man and a woman are sitting comfortably on t...
5,A sheepdog is grouping a herd of sheep
6,A man is opening a small package that contains...
7,The man in a t-shirt dyed grey is sitting on a...
8,A woman is not slicing a cucumber
9,A purple crowd of people is eating on various ...


In [15]:
import os

import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

EMBEDDING_DIM = 768  # choose 768, 1536 or 3072


def get_embedding(sentence, task_type="SEMANTIC_SIMILARITY"):
    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=sentence,
        config=types.EmbedContentConfig(
            task_type=task_type,
            output_dimensionality=EMBEDDING_DIM,
        ),
    )
    vec = np.array(result.embeddings[0].values)
    return (vec / np.linalg.norm(vec)).tolist()  # normalize

In [16]:
len(get_embedding("headphones"))

768

In [17]:
dataset["embedding"] = dataset["sentence"].apply(get_embedding)
dataset.to_csv("embedded_sentences.csv", index=False)

In [18]:
import ast

CACHE_FILE = "embedded_sentences.csv"

if os.path.exists(CACHE_FILE):
    dataset = pd.read_csv(CACHE_FILE)
    dataset["embedding"] = dataset["embedding"].apply(ast.literal_eval)
else:
    dataset["embedding"] = dataset["sentence"].apply(get_embedding)
    dataset.to_csv(CACHE_FILE, index=False)

In [19]:
dataset["id"] = range(1, len(dataset) + 1)

In [20]:
len(dataset.iloc[0]["embedding"])   # 768

768

In [21]:
embeddings = np.array(dataset.embedding.tolist())

In [22]:
query = "I love soccer"
xq = get_embedding(query)

In [23]:
def search(query, k=5):
    xq = np.array(get_embedding(query))
    scores = embeddings @ xq
    top_idx = np.argsort(scores)[::-1][:k]
    return dataset.iloc[top_idx].assign(score=scores[top_idx])[["id","sentence","score"]]

In [24]:
search("A man is drawing on a digital dry erase board", 5)

,id,sentence,score
1,2,A man is drawing on a digital dry erase board,1.000000
6,7,A man is opening a small package that contains...,0.758045
3,4,An elderly man is sitting on a bench,0.735233
7,8,The man in a t-shirt dyed grey is sitting on a...,0.728242
4,5,A man and a woman are sitting comfortably on t...,0.702261


pinecone

In [25]:
import os
from pinecone import Pinecone

PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")
database = Pinecone(api_key=PINECONE_API_KEY)

In [26]:
from pinecone import ServerlessSpec
serverless_spec = ServerlessSpec(cloud="aws", region="us-east-1")

In [56]:
INDEX_NAME = "banoqabil-fm"

if INDEX_NAME in database.list_indexes().names():
    database.delete_index(INDEX_NAME)
    time.sleep(5)

database.create_index(name=INDEX_NAME, dimension=768, metric="cosine", spec=serverless_spec)

while not database.describe_index(INDEX_NAME).status["ready"]:
    time.sleep(1)

pinecone_index = database.Index(INDEX_NAME)
print(database.describe_index(INDEX_NAME))

IndexModel(name='banoqabil-fm', status='Ready', host='https://banoqabil-fm-96jfo9v.svc.aped-4627-b74a.pinecone.io', deployment='Managed', deletion_protection='disabled', schema_fields=2)


In [ ]:
pinecone_index.describe_index_stats()

DescribeIndexStatsResponse(dimension=768, total_vector_count=0, metric='cosine', namespaces=0)

In [ ]:
def iterator(dataset, size): ...
def vector(batch): ...

In [57]:
pinecone_index.upsert(vectors=vector(dataset))

UpsertResponse(upserted_count=10)

In [60]:
pinecone_index.describe_index_stats()

DescribeIndexStatsResponse(dimension=768, total_vector_count=10, metric='cosine', namespaces=1)

In [61]:
response = pinecone_index.query(vector=xq, top_k=4, include_metadata=True)
for match in response["matches"]:
    print(match["metadata"]["sentence"])

A little girl is smiling and running outside
A sheepdog is grouping a herd of sheep
A man and a woman are sitting comfortably on the bench
The man in a t-shirt dyed grey is sitting on a rock in front of a waterfall


In [64]:
query2 = "a MAN IS WORKING"
xq2 = get_embedding(query2)
response = pinecone_index.query(vector=xq2, top_k=5, include_metadata=True)

for match in response["matches"]:
    print(f"{match['score']:.4f}  {match['metadata']['sentence']}")

0.8203  A man is drawing on a digital dry erase board
0.8158  A man is opening a small package that contains headphones
0.7909  The man in a t-shirt dyed grey is sitting on a rock in front of a waterfall
0.7903  An elderly man is sitting on a bench
0.7499  A sheepdog is grouping a herd of sheep


In [65]:
dataset.to_pickle("dataset_with_embeddings.pkl")